# Supervised Fine-Tuning
Supervised Fine-Tuning (SFT) is the first stage in the Reinforcement Learning from Human Feedback (RLHF) pipeline. In this step, a pretrained language model is trained on high-quality prompt–response pairs to learn desirable behavior and generate coherent, helpful outputs.

SFT provides a strong initial model that mimics human-like responses, which is then further refined in later stages using reward models and optimization techniques.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

**Question and Answering and formatting them**

In [ ]:
data = [
  {
    "question": "What is machine learning?",
    "answer": "Machine learning is a way for computers to learn patterns from data without being explicitly programmed."
  },
  {
    "question": "Explain reinforcement learning simply",
    "answer": "Reinforcement learning is when an agent learns by trying actions and getting rewards or penalties."
  }
]

In [ ]:
def format_example(example):
    return f"Question: {example['question']}\nAnswer: {example['answer']}"

texts = [format_example(x) for x in data]

**Encoding**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Fix
tokenizer.pad_token = tokenizer.eos_token

def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

encodings = tokenize(texts)

**Padding**

In [ ]:
print(tokenizer.pad_token)

In [ ]:
print(tokenizer.eos_token)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

**Collation with padding**

**Decoding**

In [ ]:
decoded_texts = tokenizer.batch_decode(
    encodings["input_ids"],
    skip_special_tokens=True
)

for text in decoded_texts:
    print(text)

**Creating a PyTorch Dataset for Supervised Fine-Tuning (SFT)**                                                                              
This code defines a custom dataset class using PyTorch to prepare tokenized text data for training a language model like GPT-2. It wraps the encoded inputs (input_ids and attention_mask) into a dataset format compatible with PyTorch training pipelines.

Each data sample includes:

input_ids: the tokenized representation of the text
attention_mask: indicates which tokens are actual data vs padding
labels: set equal to input_ids for causal language modeling (next-token prediction)

By structuring the data this way, the model learns to predict the next token in a sequence, which is the core objective during supervised fine-tuning.

In [ ]:
import torch

class QADataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.encodings["input_ids"][idx])
        }

dataset = QADataset(encodings)

**Padding with Collation**

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# this will dynamically pad input_ids + attention_mask
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

dataloader_params = {
    "batch_size": 16,
    "shuffle": True,
    "collate_fn": data_collator
}

train_dataloader = DataLoader(dataset, **dataloader_params)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    learning_rate=5e-5
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

In [ ]:
trainer.save_model("./sft_model")
tokenizer.save_pretrained("./sft_model")

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="./sft_model", tokenizer=tokenizer)

print(pipe("What is machine learning?", max_new_tokens=100))

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    inputs = {k: v.to(device) for k, v in inputs.items()}

    output = model.generate(**inputs, max_new_tokens=100)

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "What is machine learning?",
    "Explain reinforcement learning simply",
    "What is overfitting?",
    "What is gradient descent?"
]